In [ ]:
pip install --upgrade azure-search-documents azure-core

# 03_build_index.ipynb\n\n- `data/processed/movies_index_ready.json`: cognitive 5개 필드가 `null`인 준비 파일\n- `data/processed/movies_index_enriched.json`: cognitive 5개 필드를 채운 뒤 Azure AI Search 업로드에 쓰는 후속 입력 파일\n- 현재 enriched JSON은 `pacing`, `plot_complexity`, `emotional_intensity`, `visual_score`, `sentiment_score` 같은 원시 cognitive 값 중심이며, 파생 필드(`plot_complexity_level`, `visual_level`, `pacing_score`)는 필요 시 후속 단계에서 추가 생성\n

In [ ]:
from pathlib import Path\n\nPROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()\nINDEX_READY_PATH = PROJECT_ROOT / "data" / "processed" / "movies_index_ready.json"\nINDEX_ENRICHED_PATH = PROJECT_ROOT / "data" / "processed" / "movies_index_enriched.json"\n\nprint(f"ready source: {INDEX_READY_PATH}")\nprint(f"enriched source for upload: {INDEX_ENRICHED_PATH}")\n

In [ ]:
# ============================\n# 03_build_index.ipynb\n# Azure AI Search 인덱스 생성 + 스키마 설정\n# Upload input should be data/processed/movies_index_enriched.json\n# ============================\n\nimport os\nfrom dotenv import load_dotenv\nfrom azure.search.documents.indexes import SearchIndexClient\nfrom azure.search.documents.indexes.models import (\n    SearchIndex, SimpleField, SearchableField, SearchField,\n    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,\n    VectorSearchVectorizer, AzureOpenAIVectorizer, SemanticConfiguration,\n    SemanticPrioritizedFields, SemanticField, SemanticSearch\n)\nfrom azure.core.credentials import AzureKeyCredential\n\n# 1. .env 로드\nload_dotenv(PROJECT_ROOT / ".env")\n\nendpoint = os.getenv("AZURE_SEARCH_ENDPOINT")\nkey = os.getenv("AZURE_SEARCH_API_KEY")\nindex_name = os.getenv("AZURE_SEARCH_INDEX_NAME", "movies-index")\n\ncredential = AzureKeyCredential(key)\nclient = SearchIndexClient(endpoint=endpoint, credential=credential)\n\n# 2. 인덱스 스키마 정의\n# ready 파일은 cognitive null 상태의 준비 파일이며, 실제 업로드는 enriched 파일 기준으로 진행한다.\nfields = [\n    SimpleField(name="id", type="Edm.String", key=True, sortable=True),\n    SearchableField(name="title", type="Edm.String", analyzer_name="ko.lucene"),\n    SearchableField(name="title_ko", type="Edm.String", analyzer_name="ko.lucene"),\n    SimpleField(name="year", type="Edm.Int32", filterable=True, sortable=True),\n    SimpleField(name="runtime", type="Edm.Int32", filterable=True, sortable=True),\n    SimpleField(name="rating_imdb", type="Edm.Double", filterable=True, sortable=True),\n    SearchableField(name="overview", type="Edm.String", analyzer_name="ko.lucene"),\n    SearchField(\n        name="overview_vector",\n        type="Collection(Edm.Single)",\n        searchable=True,\n        vector_search_dimensions=1536,\n        vector_search_profile_name="movie-vector-profile"\n    ),\n    SearchableField(name="director", type="Edm.String", filterable=True),\n    SimpleField(name="tmdb_id", type="Edm.Int32"),\n    SimpleField(name="pacing", type="Edm.String", filterable=True),\n    SimpleField(name="plot_complexity", type="Edm.Int32", filterable=True),\n    SimpleField(name="emotional_intensity", type="Edm.Int32", filterable=True),\n    SimpleField(name="visual_score", type="Edm.Double", filterable=True),\n    SimpleField(name="sentiment_score", type="Edm.Double", filterable=True),\n    SearchableField(name="genres", type="Collection(Edm.String)", filterable=True, facetable=True),\n]\n\n# 3. Vector Search 설정 (HNSW + OpenAI embedding)\nvector_search = VectorSearch(\n    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],\n    profiles=[\n        VectorSearchProfile(\n            name="movie-vector-profile",\n            algorithm_configuration_name="hnsw-config",\n            vectorizer_name="openai-vectorizer"\n        )\n    ],\n    vectorizers=[\n        AzureOpenAIVectorizer(\n            name="openai-vectorizer",\n            azure_open_ai_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),\n            azure_open_ai_key=os.getenv("AZURE_OPENAI_API_KEY"),\n            azure_open_ai_deployment_name=os.getenv("AZURE_OPENAI_DEPLOY_EMBED", "text-embedding-3-small")\n        )\n    ]\n)\n\n# 4. Semantic Search 설정\nsemantic_config = SemanticConfiguration(\n    name="movie-semantic-config",\n    prioritized_fields=SemanticPrioritizedFields(\n        title_field=SemanticField(field_name="title_ko"),\n        content_fields=[SemanticField(field_name="overview")],\n        keywords_fields=[SemanticField(field_name="genres"), SemanticField(field_name="director")]\n    )\n)\nsemantic_search = SemanticSearch(configurations=[semantic_config])\n\n# 5. 인덱스 생성\nindex = SearchIndex(\n    name=index_name,\n    fields=fields,\n    vector_search=vector_search,\n    semantic_search=semantic_search\n)\n\n# 6. 인덱스 생성 또는 업데이트\ntry:\n    client.create_or_update_index(index)\n    print(f"✅ 성공! 인덱스 '{index_name}'가 생성되었습니다.")\nexcept Exception as e:\n    print(f"❌ 오류: {e}")\n